In [ ]:
test

In [ ]:
%%sql

SELECT
    form_ans_bridge_src_id,
    form_ans_bridge_src_sys_inst_src_id,
    form_ans_bridge_form_ques_src_name,
    COUNT(*) AS bridge_rows,
    COUNT(DISTINCT form_ans_bridge_form_ans_conformed_answer) AS distinct_conformed_answers,
    COUNT(DISTINCT form_ans_bridge_form_ques_src_form_type) AS distinct_form_types,
    COUNT(DISTINCT form_ans_bridge_form_ques_src_status) AS distinct_statuses
FROM silver_rdm_form_answer_bridging
WHERE form_ans_bridge_src_sys_inst_src_id LIKE 'SONE%'
GROUP BY
    form_ans_bridge_src_id,
    form_ans_bridge_src_sys_inst_src_id,
    form_ans_bridge_form_ques_src_name
HAVING COUNT(*) > 1
ORDER BY bridge_rows DESC;

In [ ]:
%%sql

WITH br_dedup AS (
    SELECT
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name,
        MAX(form_ans_bridge_form_ans_conformed_answer) AS form_ans_bridge_form_ans_conformed_answer
    FROM silver_rdm_form_answer_bridging
    WHERE form_ans_bridge_src_sys_inst_src_id LIKE 'SONE%'
    GROUP BY
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name
)

SELECT
    sc.ctv3_code,
    sc.ctv3_text,
    sc.id_organisation_source,
    CONCAT('SONE', sc.id_organisation_source) AS sc_source_instance,
    COALESCE(selc.question_heading, dermc.question_heading) AS src_question,

    br.form_ans_bridge_src_id,
    br.form_ans_bridge_src_sys_inst_src_id,
    br.form_ans_bridge_form_ques_src_name,
    br.form_ans_bridge_form_ans_conformed_answer,

    CONCAT_WS(' - ', sc.ctv3_code, br.form_ans_bridge_form_ans_conformed_answer) AS form_ans_src_answer
FROM silver_sone_srcode sc
LEFT JOIN silver_rdm_derm_read_codes dermc
    ON sc.ctv3_code = dermc.code
LEFT JOIN silver_rdm_sel_read_codes selc
    ON sc.ctv3_code = selc.code
LEFT JOIN br_dedup br
    ON TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
   AND TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
   AND TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))
WHERE sc.ctv3_code IS NOT NULL
  AND br.form_ans_bridge_form_ans_conformed_answer IS NOT NULL
LIMIT 100;